# SATP Action Type Classification

This notebook implements multi-label text classification for identifying action types in SATP incidents using transformer models.

## Notebook Structure

1. **Setup & Configuration** - Import libraries and configure models
2. **Data Management** - Load, clean, and split data  
3. **Model Components** - Define classes and functions
4. **Training Pipeline** - Run experiments across models and data fractions
5. **Results & Analysis** - Save results and generate visualizations

## Key Features
- Supports multiple transformer models (BERT, RoBERTa, DistilBERT, etc.)
- Tests performance across different data fractions (3% to 100%)
- Multi-label classification for action types (armed assault, bombing, infrastructure, etc.)
- Comprehensive evaluation with Hamming Loss, Micro F1, and Macro F1 metrics
- Automated result saving and visualization generation

## 1. Setup and Configuration

### 1.1 Environment Setup

#### 1.11 Colab Setup

In [ ]:
# 1) Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# 2) Clone or update the repo

# Choose branch once per session
BRANCH = "imbalance-handling-strategies"  # or "main"

# Clone fresh each session 
!rm -rf /content/code-satp
!git clone -b $BRANCH --depth 1 https://github.com/eteitelbaum/code-satp.git /content/code-satp

# 3) Install dependencies from the cloned repo
!pip install -qU pip setuptools wheel
!pip install -r /content/code-satp/models/classification-models/requirements.txt

# 4) Make result dirs and add to path
import pathlib
import sys
pathlib.Path("/content/drive/MyDrive/colab/satp-results").mkdir(parents=True, exist_ok=True)
pathlib.Path("/content/drive/MyDrive/colab/satp-results/action-type").mkdir(parents=True, exist_ok=True)
sys.path.append("/content/code-satp/models/classification-models")

# 5) GPU check
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# 6) Set canonical task name for this notebook
TASK_NAME = "action-type"

#### 1.12 Local Setup

In [ ]:
# # For local development, uncomment these lines:
# import sys
# sys.path.append('./utils')  # Add utils to path

# # Local data and results paths
# DATA_PATH = "../../data/"
# RESULTS_PATH = "./results/"

# # Create local results directory
# import os
# os.makedirs(RESULTS_PATH, exist_ok=True)

# # Verify GPU availability
# import torch
# if torch.cuda.is_available():
#     print(f"✅ GPU Available: {torch.cuda.get_device_name(0)}")
# else:
#     print("⚠️ GPU not available, using CPU")

# print("✅ Local setup complete!")

## 1.2 Import Libraries

In [ ]:
# Core libraries
import os
import shutil
import sys
import pandas as pd
from github import Github

# Machine learning
import torch
from torch.utils.data import Dataset
from sklearn.metrics import classification_report, hamming_loss, accuracy_score
from skmultilearn.model_selection import IterativeStratification

### Custom ML modules
sys.path.append('./utils')
from utils import (
    create_fixed_splits, 
    MultiLabelDataset, 
    compute_metrics, 
    train_transformer_model, 
    run_model_experiments
)

# Transfomers
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    Trainer, 
    TrainingArguments
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

### Custom visualization modules
from utils import scatter_plot_speed_vs_accuracy, heatmap_label_f1_scores

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Unified file I/O helpers
from utils.file_io import (
    save_dataframe_csv, load_dataframe_csv,
    get_task_results_dir, build_filename, normalize_task_name
)

## 2. Store Model Names and Labels

In [ ]:
# List of Hugging Face model identifiers to evaluate
model_names = [
    "bert-base-cased",
    "snowood1/ConfliBERT-scr-cased",
    "FacebookAI/roberta-base",
    "distilbert-base-cased",
    "xlnet-base-cased",
    "google/electra-base-discriminator"
]

# Mapping from model name to human-readable label
model_labels = {
    "bert-base-cased": "BERT",
    "snowood1/ConfliBERT-scr-cased": "ConfliBERT",
    "FacebookAI/roberta-base": "RoBERTa",
    "distilbert-base-cased": "DistilBERT",
    "xlnet-base-cased": "XLNet",
    "google/electra-base-discriminator": "ELECTRA"
}

## 3. Create Training, Validation and Test Sets

### 3.1 Read in Data

In [ ]:
# Read data from cloned repository
try:
    data = pd.read_csv('/content/code-satp/data/action_type.csv')
    print("✅ Data loaded successfully from cloned repository")
    print(data.head())
except Exception as e:
    print(f"❌ Error loading CSV from cloned repo: {e}")
    print("🔧 Fallback: Trying GitHub URL...")

# Fallback to GitHub if cloned repo fails or if working locally
url = 'https://raw.githubusercontent.com/eteitelbaum/code-satp/main/data/action_type.csv'
try:
    data = pd.read_csv(url)
    print("✅ Data loaded successfully from GitHub")
    print(data.head())
except Exception as e:
    print(f"❌ Error loading CSV from GitHub: {e}")
    

### 3.2 Create Data Splits

In [ ]:
# Define your multi-label target columns
stratify_cols = [  
    "armed_assault", "arrest", "bombing", "infrastructure",
    "surrender", "seizure", "abduction"
]

# Create fixed splits (once!)
df_train_pool, df_val, df_test = create_fixed_splits(
    df_full=data,
    stratify_cols=stratify_cols,
    test_size=0.1,
    val_size=0.111  # ~10% of original
)

### 3.3 Verify Size of Training, Test and Validation Sets

In [ ]:
print("Total rows:", len(data))
print("Training pool:", len(df_train_pool))
print("Validation set:", len(df_val))
print("Test set:", len(df_test))

Total rows: 9921
Test set: 972
Training pool: 7955
Validation set: 994


### 3.4 Summarize Support Across Three Sets

In [ ]:
print(df_train_pool[stratify_cols].sum())
print(df_val[stratify_cols].sum())
print(df_test[stratify_cols].sum())

## 4. Run the Models

### 4.1 Run Experiments

In [ ]:
test_results_df, test_predictions_df = run_model_experiments(
    df_train_pool, df_val, df_test,
    model_names=model_names,
    model_labels=model_labels,
    stratify_cols=stratify_cols,
    output_csv="test_summary.csv",
    predictions_csv="test_predictions.csv"
)

### 4.2 Save Results

In [ ]:
# Unified save using helpers
save_dataframe_csv(test_results_df, "actiontype_test1_summary.csv", task_name=TASK_NAME)
save_dataframe_csv(test_predictions_df, "actiontype_test1_predictions.csv", task_name=TASK_NAME)

### 4.3 Verify File Creation

In [ ]:
# List files using Python for portability
import os
print("\n".join(sorted(os.listdir(get_task_results_dir(TASK_NAME)))))

## 5. Preliminary Analysis

### 5.1 Read in Data

In [ ]:
df = load_dataframe_csv('actiontype_test1_summary.csv', task_name=TASK_NAME)

### 5.2 F1 Micro

In [ ]:
# Create the plot
plt.figure(figsize=(10, 6))
sns.lineplot(
    data=df,
    x="fraction_label",
    y="eval_micro avg_f1-score",
    hue="model_label",
    marker="o",
    palette="turbo"
    )

# Customize the plot
plt.xlabel("Fraction of Data")
plt.ylabel("Micro F1 Score")
plt.title("Model Peformance for 'Action Type' Classification Tasks")
plt.xticks(rotation=45, ha='right')  # Rotate x-axis labels for better readability
plt.tight_layout()
plt.show()

### 5.3 Heatmap of Individual Labels

In [ ]:
heatmap_label_f1_scores(
      df=df,
      title="F1 Scores for Each Target Type Across Models (100% Data)",
      note="Models sorted by average Micro F1 Score and action type sorted by average label F1 score."
)

### 5.4 Model Performance vs. Speed

In [ ]:
scatter_plot_speed_vs_accuracy(
    df=df,
    x_col="eval_samples_per_second",
    y_col="eval_micro avg_f1-score",
    hue_col="model_label",
    size_col="fraction_raw",
    title="Model Performance vs. Speed for 'Action Type' Evaluation Task"
)

## 6. Imbalance Handling Strategies

### 6.1 Setup

In [ ]:
from utils import run_strategy_experiments, heatmap_label_f1_by_strategy

MODEL_NAME = "snowood1/ConfliBERT-scr-cased"
STRATEGIES = [
  "baseline", "focal", "class_weights",
  "threshold_tuned", "weighted_sampler", "augmentation_bt", 
  "augmentation_t5"
]
LABEL_COLS = stratify_cols
RESULTS_FOLDER = "action-type"  

### 6.2 Run Strategy Sweep

In [ ]:
pivot, results_long = run_strategy_experiments(
    df_train=df_train_pool,
    df_val=df_val,
    df_test=df_test,
    label_cols=LABEL_COLS,
    model_name=MODEL_NAME,
    strategies=STRATEGIES,
    max_len=512,
    batch_size=16,
    epochs=2
)

### 6.3 Save Results

In [ ]:
# Save strategy results using unified helpers
save_dataframe_csv(pivot.reset_index(), f"{RESULTS_FOLDER}_strategy_f1_pivot.csv", task_name=TASK_NAME)
save_dataframe_csv(results_long, f"{RESULTS_FOLDER}_strategy_f1_long.csv", task_name=TASK_NAME)

### 6.4 Verify File Creation

In [ ]:
# List files using Python for portability
import os
print("\n".join(sorted(os.listdir(get_task_results_dir(TASK_NAME)))))

## 7. Visualize Imbalance Handling Strategies

### 7.1 Read in Results

In [ ]:
df_imbalance = load_dataframe_csv('action-type_strategy_f1_long.csv', task_name=TASK_NAME)

### 7.2 Heatmap of F1 by Label and Strategy

In [ ]:
heatmap_label_f1_by_strategy(
    results_long,
    title=f"F1 by Label vs Imbalance Strategy (ConfliBERT, 100% train) – {RESULTS_FOLDER}"
)